Explanation

This cell installs all required libraries for the project. The goal is to keep the system fully runnable in Google Colab without relying on paid APIs or unstable external services.

In [ ]:
!pip install -q transformers sentence-transformers faiss-cpu lime nltk scikit-learn pandas numpy


Explanation

This cell imports all core tools. The system combines classical NLP, transformer-based embeddings, retrieval, calibration, and explainability. IsotonicRegression is added to calibrate confidence scores, which makes the final confidence more trustworthy and more professional.

In [ ]:
import re
import string
import warnings
from dataclasses import dataclass, field
from collections import defaultdict
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import torch
import faiss
import nltk

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
    top_k_accuracy_score
)
from sklearn.isotonic import IsotonicRegression

from transformers import AutoTokenizer, AutoModel
from sentence_transformers import SentenceTransformer
from lime.lime_text import LimeTextExplainer

warnings.filterwarnings("ignore")
nltk.download("punkt", quiet=True)
nltk.download("stopwords", quiet=True)


True

In [ ]:
from dataclasses import dataclass, field
from sklearn.metrics import precision_recall_fscore_support


In [ ]:
@dataclass
class CaseMemory:
    main_complaint: str = ""
    body_part: List[str] = field(default_factory=list)
    severity: str = "unknown"
    duration: str = "unknown"
    spread: str = "unknown"
    red_flags: List[str] = field(default_factory=list)
    follow_up_answers: List[Dict[str, str]] = field(default_factory=list)


In [ ]:
@dataclass
class PatientProfile:
    age_group: str = "unknown"
    sex: str = "unknown"
    chronic_conditions: str = "unknown"


In [ ]:
@dataclass
class SystemConfig:
    text_col: str = "question_body"
    label_col: str = "name_ar"

    random_state: int = 42
    test_size: float = 0.15
    val_size: float = 0.15

    arabert_model_name: str = "aubmindlab/bert-base-arabertv2"
    arabert_batch_size: int = 16
    arabert_max_length: int = 128

    retrieval_model_name: str = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
    retrieval_balance_per_label: int = 100
    faiss_candidates: int = 20
    top_k_retrieval: int = 5
    dense_weight: float = 0.75
    lexical_weight: float = 0.25

    lime_num_features: int = 8
    use_gpu_if_available: bool = True

    uncertainty_threshold: float = 0.60
    max_follow_up_questions: int = 10


config = SystemConfig()
device = "cuda" if config.use_gpu_if_available and torch.cuda.is_available() else "cpu"
print("Device:", device)


Device: cuda


Explanation

This cell centralizes all tunable settings in one place. This is a professional design practice because it improves reproducibility, readability, and experimentation. It also makes the project easier to explain to a supervisor.

In [ ]:
# Replace with your actual file path in Colab
df = pd.read_csv('altibbi_specialty_data.csv')

df = df[[config.text_col, config.label_col]].dropna().copy()
df["text"] = df[config.text_col].astype(str)
df["label"] = df[config.label_col].astype(str)

print(df.shape)
df.head()


(92559, 4)


,question_body,name_ar,text,label
0,استشاره عيون,طب عيون,استشاره عيون,طب عيون
1,السلام عليكم ممكن دكتور مفاصل واعصاب,جراحة العظام والمفاصل,السلام عليكم ممكن دكتور مفاصل واعصاب,جراحة العظام والمفاصل
2,عندي نقص فيتامين د هل ممكن استخدم معه كالسيوم,جراحة العظام والمفاصل,عندي نقص فيتامين د هل ممكن استخدم معه كالسيوم,جراحة العظام والمفاصل
3,عمليه الحول للكبار,طب عيون,عمليه الحول للكبار,طب عيون
4,ألم بالكتف الايسر من فترة,جراحة العظام والمفاصل,ألم بالكتف الايسر من فترة,جراحة العظام والمفاصل


Explanation

This cell loads the dataset and keeps only the two relevant columns: the symptom text and the target specialty. This keeps the pipeline focused and avoids unnecessary memory usage.

In [ ]:
nltk.download("punkt_tab", quiet=True)

class ArabicPreprocessor:
    def __init__(self):
        self.arabic_stopwords = set(stopwords.words("arabic"))
        self.punct_table = str.maketrans("", "", string.punctuation)

    @staticmethod
    def normalize_arabic(text: str) -> str:
        text = str(text)
        text = re.sub(r"[إأآa]", "ا", text)
        text = re.sub(r"ى", "ي", text)
        text = re.sub(r"ؤ", "و", text)
        text = re.sub(r"ئ", "ي", text)
        text = re.sub(r"ة", "ه", text)
        text = re.sub(r"ء", "", text)
        text = re.sub(r"[\u064B-\u065F]", "", text)
        text = re.sub(r"[^\u0600-\u06FF0-9\s]", " ", text)
        text = re.sub(r"\s+", " ", text).strip()
        return text

    def preprocess(self, text: str) -> str:
        text = self.normalize_arabic(text)
        text = text.translate(self.punct_table)
        tokens = word_tokenize(text)
        tokens = [t for t in tokens if t not in self.arabic_stopwords and len(t) > 1]
        return " ".join(tokens)

    @staticmethod
    def tokenize(text: str) -> List[str]:
        return str(text).split()


preprocessor = ArabicPreprocessor()
df["clean"] = df["text"].apply(preprocessor.preprocess)
df = df[df["clean"].str.strip().astype(bool)].reset_index(drop=True)

print(df.shape)
df[["text", "clean", "label"]].head()

(92347, 5)


,text,clean,label
0,استشاره عيون,استشاره عيون,طب عيون
1,السلام عليكم ممكن دكتور مفاصل واعصاب,السلام عليكم ممكن دكتور مفاصل واعصاب,جراحة العظام والمفاصل
2,عندي نقص فيتامين د هل ممكن استخدم معه كالسيوم,عندي نقص فيتامين ممكن استخدم معه كالسيوم,جراحة العظام والمفاصل
3,عمليه الحول للكبار,عمليه الحول للكبار,طب عيون
4,ألم بالكتف الايسر من فترة,الم بالكتف الايسر فتره,جراحة العظام والمفاصل


Explanation

Arabic preprocessing is especially important because spelling variants, diacritics, and character forms can reduce model quality. This step normalizes the text and removes noise so both the classifier and retriever operate on cleaner symptom descriptions.

In [ ]:
class SymptomExtractor:
    def __init__(self, preprocessor: ArabicPreprocessor):
        self.preprocessor = preprocessor

        self.body_parts = {
            "صدر": ["صدر", "قلب", "قلبي"],
            "راس": ["راس", "صداع", "دماغ"],
            "رقبه": ["رقبه", "عنق"],
            "كتف": ["كتف"],
            "ظهر": ["ظهر", "عمود"],
            "بطن": ["بطن", "معده", "مغص"],
            "رجل": ["رجل", "ساق", "قدم"],
            "يد": ["يد", "ذراع", "كف"],
        }

        self.severity_terms = {
            "mild": ["خفيف", "بسيط"],
            "moderate": ["متوسط", "متوسطة"],
            "severe": ["شديد", "حاد", "قوي", "لا يحتمل"],
        }

        self.duration_terms = {
            "acute": ["اليوم", "منذ ساعه", "منذ يوم", "فجاه", "مفاجئ"],
            "subacute": ["منذ ايام", "منذ اسبوع", "من اسبوع", "منذ اسابيع", "من فترة"],
            "chronic": ["منذ شهر", "منذ شهور", "مزمن", "منذ فتره طويله"],
        }

        self.spread_terms = ["يمتد", "ينتشر", "للذراع", "الى الذراع", "إلى الذراع", "للفك", "للظهر"]
        self.red_flags = ["اغماء", "نزيف", "شلل", "ضيق تنفس", "قيء دم", "سعال دم", "تنميل", "ضعف", "زغلله", "تعرق"]
        self.fever_terms = ["حراره", "حرارة", "سخونه", "سخونة"]
        self.weakness_terms = ["ضعف", "لا استطيع", "لا أستطيع", "عدم القدره", "عدم القدرة"]
        self.blurred_vision_terms = ["زغلله", "تشوش", "عدم وضوح"]
        self.sweating_terms = ["تعرق", "عرق", "بارد"]

    def extract(self, text: str) -> Dict[str, object]:
        clean = self.preprocessor.preprocess(text)

        body_parts = [
            name for name, keywords in self.body_parts.items()
            if any(keyword in clean for keyword in keywords)
        ]

        severity = "unknown"
        for level, keywords in self.severity_terms.items():
            if any(keyword in clean for keyword in keywords):
                severity = level
                break

        duration = "unknown"
        for level, keywords in self.duration_terms.items():
            if any(keyword in clean for keyword in keywords):
                duration = level
                break

        spread = "yes" if any(term in clean for term in self.spread_terms) else "no"
        red_flags = [flag for flag in self.red_flags if flag in clean]

        return {
            "clean_text": clean,
            "body_parts": body_parts,
            "severity": severity,
            "duration": duration,
            "spread": spread,
            "red_flags": red_flags,
            "has_fever": any(term in clean for term in self.fever_terms),
            "has_weakness": any(term in clean for term in self.weakness_terms),
            "has_blurred_vision": any(term in clean for term in self.blurred_vision_terms),
            "has_sweating": any(term in clean for term in self.sweating_terms),
        }


symptom_extractor = SymptomExtractor(preprocessor)


In [ ]:
class SafetyLayer:
    def __init__(self, preprocessor: ArabicPreprocessor):
        self.preprocessor = preprocessor
        self.direct_red_flags = [
            "فقدان الوعي",
            "نزيف شديد",
            "قيء دم",
            "سعال دم",
            "تشنج",
            "اختلاج",
            "شلل مفاجئ",
            "عدم القدره الكلام",
            "عدم القدرة الكلام",
            "ضيق تنفس شديد",
            "الم صدر شديد",
            "الم صدر مع تعرق",
            "صداع شديد جدا",
        ]

    def screen(self, text: str, extracted: Optional[Dict[str, object]] = None) -> Optional[Dict[str, str]]:
        clean = extracted["clean_text"] if extracted else self.preprocessor.preprocess(text)

        for phrase in self.direct_red_flags:
            if phrase in clean:
                return {
                    "level": "emergency",
                    "reason": f"تم رصد علامة خطر مباشرة: {phrase}",
                    "advice": "يرجى التوجه إلى الطوارئ أو طلب الإسعاف فوراً.",
                }

        if extracted:
            body_parts = extracted.get("body_parts", [])

            if "صدر" in body_parts and ("ضيق تنفس" in clean or extracted.get("has_sweating")):
                return {
                    "level": "urgent",
                    "reason": "تم رصد ألم صدري مع ضيق تنفس أو تعرق.",
                    "advice": "يفضل تقييم الحالة اليوم، والطوارئ إذا اشتدت الأعراض."
                }

            if "كتف" in body_parts and extracted.get("has_weakness") and "تنميل" in clean:
                return {
                    "level": "urgent",
                    "reason": "تم رصد ألم كتف مع تنميل وضعف.",
                    "advice": "يفضل تقييم الحالة خلال نفس اليوم."
                }

            if "راس" in body_parts and extracted.get("has_blurred_vision"):
                return {
                    "level": "urgent",
                    "reason": "تم رصد صداع مع زغللة أو تشوش رؤية.",
                    "advice": "يفضل تقييم الحالة طبياً اليوم."
                }

            if extracted.get("red_flags"):
                return {
                    "level": "urgent",
                    "reason": f"تم رصد مؤشرات خطر: {', '.join(extracted['red_flags'])}",
                    "advice": "يفضل التقييم الطبي السريع."
                }

        return None


safety_layer = SafetyLayer(preprocessor)


Explanation

This is a major upgrade. Instead of treating the input only as raw text, the system now extracts structured clinical clues such as body part, severity, duration, and red flags. This makes the chatbot, recommendation engine, and safety layer much smarter.

Explanation

This layer ensures the system behaves responsibly. A professional triage system must prioritize safety before classification. If dangerous patterns are detected, the system escalates immediately instead of continuing normal recommendation flow.

Explanation

This split is very important. The retriever and classifier must be trained only on training data. Validation data is used to tune thresholds and calibration. Test data is reserved for final reporting. This prevents data leakage and makes the evaluation scientifically correct.

In [ ]:
train_df, temp_df = train_test_split(
    df,
    test_size=config.test_size + config.val_size,
    stratify=df["label"],
    random_state=config.random_state,
)

relative_val = config.val_size / (config.test_size + config.val_size)

val_df, test_df = train_test_split(
    temp_df,
    test_size=1 - relative_val,
    stratify=temp_df["label"],
    random_state=config.random_state,
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("Train:", train_df.shape)
print("Val:", val_df.shape)
print("Test:", test_df.shape)


Train: (64642, 5)
Val: (13852, 5)
Test: (13853, 5)


Explanation

Baselines are essential in academic work. They show that the advanced hybrid system is not used blindly. Instead, the project demonstrates comparison against simpler methods and justifies the final architecture.

In [ ]:
class BaselineEvaluator:
    def __init__(self):
        self.vectorizer = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_df=0.95)
        self.models = {
            "LogisticRegression": LogisticRegression(max_iter=1200),
            "LinearSVC": LinearSVC()
        }

    def run(self, train_df: pd.DataFrame, eval_df: pd.DataFrame) -> pd.DataFrame:
        x_train = self.vectorizer.fit_transform(train_df["clean"])
        x_eval = self.vectorizer.transform(eval_df["clean"])

        rows = []
        for name, model in self.models.items():
            model.fit(x_train, train_df["label"])
            preds = model.predict(x_eval)
            rows.append({
                "model": name,
                "accuracy": accuracy_score(eval_df["label"], preds),
                "macro_f1": f1_score(eval_df["label"], preds, average="macro")
            })
        return pd.DataFrame(rows).sort_values("macro_f1", ascending=False)


baseline_evaluator = BaselineEvaluator()
baseline_results = baseline_evaluator.run(train_df, val_df)
baseline_results


,model,accuracy,macro_f1
1,LinearSVC,0.895611,0.894748
0,LogisticRegression,0.893084,0.892793


Explanation

AraBERT is the main semantic classifier. Instead of full fine-tuning, this implementation uses AraBERT as a frozen encoder and trains a lightweight logistic regression classifier on top. This is much more stable in Colab and still very strong.

In [ ]:
class AraBERTClassifier:
    def __init__(self, config: SystemConfig):
        self.config = config
        self.device = "cuda" if config.use_gpu_if_available and torch.cuda.is_available() else "cpu"
        self.tokenizer = AutoTokenizer.from_pretrained(config.arabert_model_name)
        self.encoder = AutoModel.from_pretrained(config.arabert_model_name).to(self.device)
        self.encoder.eval()
        self.classifier = LogisticRegression(max_iter=1200, class_weight="balanced")
        self.classes_ = None

    def encode(self, texts: List[str], batch_size: Optional[int] = None) -> np.ndarray:
        batch_size = batch_size or self.config.arabert_batch_size
        all_embeddings = []

        for start in range(0, len(texts), batch_size):
            batch = texts[start:start + batch_size]
            inputs = self.tokenizer(
                batch,
                return_tensors="pt",
                truncation=True,
                padding=True,
                max_length=self.config.arabert_max_length
            )
            inputs = {k: v.to(self.device) for k, v in inputs.items()}

            with torch.no_grad():
                outputs = self.encoder(**inputs)

            cls_embeddings = outputs.last_hidden_state[:, 0, :].cpu().numpy()
            all_embeddings.append(cls_embeddings)

        return np.vstack(all_embeddings)

    def fit(self, train_texts: List[str], train_labels: List[str]):
        x_train = self.encode(train_texts)
        self.classifier.fit(x_train, train_labels)
        self.classes_ = self.classifier.classes_

    def predict_proba(self, texts: List[str]) -> np.ndarray:
        x = self.encode(texts)
        return self.classifier.predict_proba(x)

    def predict(self, texts: List[str]) -> np.ndarray:
        probs = self.predict_proba(texts)
        return self.classes_[np.argmax(probs, axis=1)]


arabert_classifier = AraBERTClassifier(config)
arabert_classifier.fit(train_df["clean"].tolist(), train_df["label"].tolist())


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: aubmindlab/bert-base-arabertv2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
bert.embeddings.position_ids               | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Explanation

This cell reports the standalone AraBERT performance. It is useful because later we will compare it with the hybrid system to verify that retrieval actually improves the model.

In [ ]:
val_probs = arabert_classifier.predict_proba(val_df["clean"].tolist())
val_preds = arabert_classifier.classes_[np.argmax(val_probs, axis=1)]

print("AraBERT Validation Accuracy:", accuracy_score(val_df["label"], val_preds))
print("AraBERT Validation Macro F1:", f1_score(val_df["label"], val_preds, average="macro"))
print(classification_report(val_df["label"], val_preds))


AraBERT Validation Accuracy: 0.7717297141207046
AraBERT Validation Macro F1: 0.7716748462784341
                       precision    recall  f1-score   support

          الطب النفسي       0.72      0.79      0.75      2788
                تغذية       0.76      0.79      0.78      2067
جراحة العظام والمفاصل       0.81      0.75      0.78      3813
             طب اسنان       0.79      0.77      0.78      2790
              طب عيون       0.77      0.77      0.77      2394

             accuracy                           0.77     13852
            macro avg       0.77      0.77      0.77     13852
         weighted avg       0.77      0.77      0.77     13852



Explanation

This is a much stronger RAG design than simple label voting. It uses dense semantic retrieval, but it also adds lexical overlap reranking so the retrieved cases are more grounded in the actual words used by the patient. It also balances the retrieval bank across specialties to reduce majority-label bias

In [ ]:
class DenseArabicRetriever:
    def __init__(self, config: SystemConfig, preprocessor: ArabicPreprocessor):
        self.config = config
        self.preprocessor = preprocessor
        self.model = SentenceTransformer(config.retrieval_model_name)
        self.index = None
        self.index_df = None

    def _balanced_subset(self, train_df: pd.DataFrame) -> pd.DataFrame:
        parts = []
        for _, group in train_df.groupby("label"):
            n = min(self.config.retrieval_balance_per_label, len(group))
            parts.append(group.sample(n=n, random_state=self.config.random_state))
        return pd.concat(parts).reset_index(drop=True)

    def build(self, train_df: pd.DataFrame):
        self.index_df = self._balanced_subset(train_df).copy()
        embeddings = self.model.encode(
            self.index_df["clean"].tolist(),
            batch_size=64,
            convert_to_numpy=True,
            normalize_embeddings=True,
            show_progress_bar=True,
        ).astype("float32")

        self.index = faiss.IndexFlatIP(embeddings.shape[1])
        self.index.add(embeddings)

    def _lexical_overlap(self, q_tokens: List[str], d_tokens: List[str]) -> float:
        if not q_tokens or not d_tokens:
            return 0.0
        q_set, d_set = set(q_tokens), set(d_tokens)
        return len(q_set & d_set) / max(len(q_set), 1)

    def retrieve(self, text: str, top_k: Optional[int] = None) -> List[Dict[str, object]]:
        top_k = top_k or config.top_k_retrieval
        clean = self.preprocessor.preprocess(text)

        q_emb = self.model.encode(
            [clean],
            convert_to_numpy=True,
            normalize_embeddings=True
        ).astype("float32")

        dense_scores, idxs = self.index.search(q_emb, config.faiss_candidates)
        q_tokens = self.preprocessor.tokenize(clean)

        candidates = []
        for dense_score, idx in zip(dense_scores[0], idxs[0]):
            row = self.index_df.iloc[int(idx)]
            d_tokens = self.preprocessor.tokenize(row["clean"])
            lexical_score = self._lexical_overlap(q_tokens, d_tokens)
            final_score = config.dense_weight * float(dense_score) + config.lexical_weight * lexical_score

            candidates.append({
                "text": row["text"],
                "clean": row["clean"],
                "label": row["label"],
                "dense_score": float(dense_score),
                "lexical_score": lexical_score,
                "final_score": final_score,
            })

        candidates.sort(key=lambda x: x["final_score"], reverse=True)
        return candidates[:top_k]

    def label_distribution(self, text: str) -> Tuple[List[Dict[str, object]], Dict[str, float]]:
        retrieved = self.retrieve(text)
        score_by_label = defaultdict(float)

        for item in retrieved:
            score_by_label[item["label"]] += max(item["final_score"], 0.0)

        total = sum(score_by_label.values()) + 1e-12
        distribution = {label: score / total for label, score in score_by_label.items()}
        return retrieved, distribution


retriever = DenseArabicRetriever(config, preprocessor)
retriever.build(train_df)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/8 [00:00<?, ?it/s]

Explanation

Raw model probabilities are often miscalibrated. In professional systems, confidence should reflect real reliability. This cell adds a simple but effective calibration stage that learns how trustworthy the predicted confidence really is.

In [ ]:
class ConfidenceCalibrator:
    def __init__(self):
        self.model = IsotonicRegression(out_of_bounds="clip")
        self.is_fitted = False

    def fit(self, confidences: np.ndarray, correct: np.ndarray):
        self.model.fit(confidences, correct)
        self.is_fitted = True

    def calibrate(self, confidence: float) -> float:
        if not self.is_fitted:
            return float(confidence)
        return float(self.model.predict([confidence])[0])


confidence_calibrator = ConfidenceCalibrator()


Explanation

This is the core intelligence layer. It combines learned semantic understanding from AraBERT with grounded evidence from retrieved historical cases. It also returns the top-2 specialties and the confidence gap, which makes the system more honest and more clinically useful.

In [ ]:
class HybridDecisionEngine:
    def __init__(self, classifier, retriever):
        self.classifier = classifier
        self.retriever = retriever
        self.all_labels = list(classifier.classes_)
        self.label_to_idx = {label: i for i, label in enumerate(self.all_labels)}

        self.bodypart_specialty_map = {
            "كتف": ["جراحة العظام والمفاصل", "طب الاعصاب", "طب عام"],
            "ظهر": ["جراحة العظام والمفاصل", "طب الاعصاب", "الباطنية", "طب عام"],
            "صدر": ["أمراض القلب", "الباطنية", "طب عام"],
            "راس": ["طب الاعصاب", "الباطنية", "طب عام"],
            "بطن": ["الباطنية", "الجهاز الهضمي", "طب عام"],
            "رقبه": ["جراحة العظام والمفاصل", "طب الاعصاب", "طب عام"],
            "يد": ["جراحة العظام والمفاصل", "طب الاعصاب", "طب عام"],
            "رجل": ["جراحة العظام والمفاصل", "طب الاعصاب", "طب عام"],
        }

    def _apply_bodypart_penalty(self, final_probs, extracted):
        body_parts = extracted.get("body_parts", [])
        if not body_parts:
            return final_probs

        allowed_specialties = set()
        for part in body_parts:
            if part in self.bodypart_specialty_map:
                allowed_specialties.update(self.bodypart_specialty_map[part])

        if not allowed_specialties:
            return final_probs

        adjusted_probs = final_probs.copy()
        for label, idx in self.label_to_idx.items():
            if label not in allowed_specialties:
                adjusted_probs[idx] *= 0.05

        adjusted_probs = adjusted_probs / adjusted_probs.sum()
        return adjusted_probs

    def predict(self, text: str, extracted: Optional[Dict[str, object]] = None) -> Dict[str, object]:
        clf_probs = self.classifier.predict_proba([text])[0]
        clf_idx = int(np.argmax(clf_probs))
        clf_label = self.all_labels[clf_idx]
        clf_conf = float(clf_probs[clf_idx])

        retrieved, rag_dist = self.retriever.label_distribution(text)
        rag_probs = np.zeros(len(self.all_labels), dtype=float)

        for label, score in rag_dist.items():
            if label in self.label_to_idx:
                rag_probs[self.label_to_idx[label]] = score

        rag_idx = int(np.argmax(rag_probs)) if rag_probs.sum() > 0 else clf_idx
        rag_label = self.all_labels[rag_idx]
        rag_conf = float(rag_probs[rag_idx]) if rag_probs.sum() > 0 else 0.0

        if rag_probs.sum() == 0:
            clf_weight, rag_weight = 0.92, 0.08
        elif clf_conf >= 0.80 and clf_label == rag_label:
            clf_weight, rag_weight = 0.78, 0.22
        elif clf_conf >= 0.70:
            clf_weight, rag_weight = 0.70, 0.30
        elif clf_conf >= 0.55:
            clf_weight, rag_weight = 0.58, 0.42
        else:
            clf_weight, rag_weight = 0.45, 0.55

        final_probs = clf_weight * clf_probs + rag_weight * rag_probs

        if clf_label == rag_label:
            final_probs[self.label_to_idx[clf_label]] += 0.05

        final_probs = final_probs / final_probs.sum()

        if extracted is not None:
            final_probs = self._apply_bodypart_penalty(final_probs, extracted)

        sorted_idx = np.argsort(final_probs)[::-1]
        top1_idx = int(sorted_idx[0])

        top2_idx = None

        if extracted is not None and extracted.get("body_parts"):
            allowed_specialties = set()
            for part in extracted["body_parts"]:
                if part in self.bodypart_specialty_map:
                    allowed_specialties.update(self.bodypart_specialty_map[part])

            for idx in sorted_idx[1:]:
                candidate_label = self.all_labels[int(idx)]
                if candidate_label != self.all_labels[top1_idx] and candidate_label in allowed_specialties:
                    top2_idx = int(idx)
                    break

        if top2_idx is None:
            for idx in sorted_idx[1:]:
                candidate_label = self.all_labels[int(idx)]
                if candidate_label != self.all_labels[top1_idx]:
                    top2_idx = int(idx)
                    break

        if top2_idx is None:
            top2_idx = top1_idx

        return {
            "final_label": self.all_labels[top1_idx],
            "final_confidence_raw": float(final_probs[top1_idx]),
            "second_label": self.all_labels[top2_idx],
            "second_confidence": float(final_probs[top2_idx]),
            "confidence_gap": float(final_probs[top1_idx] - final_probs[top2_idx]),
            "classifier_label": clf_label,
            "classifier_confidence": clf_conf,
            "retrieval_label": rag_label,
            "retrieval_confidence": rag_conf,
            "weights": {"classifier": clf_weight, "retrieval": rag_weight},
            "final_probs": final_probs,
            "retrieved_cases": retrieved,
        }


hybrid_engine = HybridDecisionEngine(arabert_classifier, retriever)


Explanation

This cell trains the confidence calibrator using validation data. The model learns how often a raw confidence value is actually correct. This makes the final reported confidence much more reliable.

In [ ]:
val_raw_confidences = []
val_correct = []

for text, true_label in zip(val_df["clean"].tolist(), val_df["label"].tolist()):
    result = hybrid_engine.predict(text)
    val_raw_confidences.append(result["final_confidence_raw"])
    val_correct.append(int(result["final_label"] == true_label))

val_raw_confidences = np.array(val_raw_confidences)
val_correct = np.array(val_correct)

confidence_calibrator.fit(val_raw_confidences, val_correct)
print("Confidence calibrator fitted.")


Confidence calibrator fitted.


Explanation

This is a more intelligent chatbot design. Instead of static questions, the system now chooses questions according to specialty, symptom pattern, severity, and model uncertainty. This makes the interaction feel much more professional.

In [ ]:
@dataclass
class QuestionTemplate:
    question: str
    category: str
    stage: str = "specialty"
    specialties: List[str] = field(default_factory=list)
    symptom_triggers: List[str] = field(default_factory=list)
    why: str = ""
    risk_weight_yes: int = 1
    risk_weight_severe: int = 2


class AdaptiveQuestionEngine:
    def __init__(self, preprocessor: ArabicPreprocessor):
        self.preprocessor = preprocessor
        self.templates = self._build_templates()

    def _build_templates(self) -> List[QuestionTemplate]:
        return [
            QuestionTemplate("كم عمرك تقريباً؟ (طفل / بالغ / كبير سن)", "age_group", stage="profile"),
            QuestionTemplate("ما الجنس؟ (ذكر / أنثى)", "sex", stage="profile"),
            QuestionTemplate("هل لديك أمراض مزمنة مثل السكري أو الضغط أو أمراض القلب؟", "chronic_disease", stage="profile"),

            QuestionTemplate("أين مكان الألم أو العرض بشكل أدق؟", "clarify_location", stage="clarify"),
            QuestionTemplate("منذ متى بدأت الأعراض؟", "clarify_duration", stage="clarify"),
            QuestionTemplate("من 0 إلى 10 كم شدة الألم؟", "pain_score", stage="clarify"),

            QuestionTemplate("هل الأعراض تتحسن، ثابتة، أم تزداد؟", "timeline", stage="general"),
            QuestionTemplate("هل يزيد الألم مع الحركة؟", "trigger_motion", stage="general"),
            QuestionTemplate("هل يخف الألم مع الراحة؟", "relief_rest", stage="general"),

            QuestionTemplate("هل يمتد الألم إلى الذراع؟", "shoulder_radiation", stage="shoulder"),
            QuestionTemplate("هل يوجد تنميل أو خدر في الذراع أو اليد؟", "shoulder_numbness", stage="shoulder"),
            QuestionTemplate("هل يوجد ضعف أو صعوبة في رفع الذراع؟", "shoulder_weakness", stage="shoulder"),
            QuestionTemplate("هل توجد إصابة سابقة أو حمل شيء ثقيل؟", "shoulder_injury", stage="shoulder"),

            QuestionTemplate("هل يوجد ألم في الصدر أو ضيق في التنفس أو تعرق بارد؟", "chest_workflow", stage="chest"),
            QuestionTemplate("هل الصداع شديد؟ وهل يوجد زغللة أو قيء أو ضعف؟", "headache_workflow", stage="headache"),
            QuestionTemplate("هل يوجد غثيان أو قيء أو حرارة أو إسهال أو إمساك؟", "abdominal_workflow", stage="abdomen"),
        ]

    def clarification_questions_needed(self, extracted: Dict[str, object], user_text: str) -> bool:
        return len(user_text.split()) <= 4 or not extracted["body_parts"] or extracted["severity"] == "unknown"

    def select_questions(
        self,
        user_text: str,
        predicted_specialty: str,
        final_confidence: float,
        extracted: Dict[str, object],
        max_questions: int = 8,
    ) -> List[QuestionTemplate]:
        clean = self.preprocessor.preprocess(user_text)
        selected = []

        selected.extend([t for t in self.templates if t.stage == "profile"][:3])

        if self.clarification_questions_needed(extracted, user_text):
            selected.extend([t for t in self.templates if t.stage == "clarify"])

        selected.extend([t for t in self.templates if t.stage == "general"][:2])

        body_parts = extracted.get("body_parts", [])

        if "كتف" in body_parts:
            selected.extend([t for t in self.templates if t.stage == "shoulder"])

        if "صدر" in body_parts:
            selected.extend([t for t in self.templates if t.stage == "chest"])

        if "راس" in body_parts:
            selected.extend([t for t in self.templates if t.stage == "headache"])

        if "بطن" in body_parts:
            selected.extend([t for t in self.templates if t.stage == "abdomen"])

        if final_confidence < 0.60:
            selected.append(
                QuestionTemplate(
                    "هل توجد أعراض أخرى مهمة لم تذكرها؟",
                    "open_symptoms",
                    stage="general",
                    why="لأن الحالة غير محسومة بالكامل ونحتاج تفاصيل إضافية.",
                    risk_weight_yes=1,
                    risk_weight_severe=1,
                )
            )

        deduped = []
        seen = set()
        for q in selected:
            if q.question not in seen:
                deduped.append(q)
                seen.add(q.question)

        return deduped[:max_questions]


question_engine = AdaptiveQuestionEngine(preprocessor)


Explanation

The recommendation engine is now based on the actual follow-up answers rather than fixed outputs. This makes the final recommendation more personalized, more interpretable, and more useful.

In [ ]:
class RecommendationEngine:
    @staticmethod
    def _answer_features(answer: str) -> Dict[str, object]:
        text = str(answer).strip().lower()

        duration = "unknown"
        if any(t in text for t in ["منذ يوم", "اليوم", "مفاجئ", "فجاه", "منذ يومين", "يومين"]):
            duration = "acute"
        elif any(t in text for t in ["منذ اسبوع", "من اسبوع", "منذ أسبوع", "أسبوع", "اسبوع"]):
            duration = "subacute"
        elif any(t in text for t in ["منذ شهر", "منذ فتره", "مزمن"]):
            duration = "chronic"

        severity = "unknown"
        if any(t in text for t in ["خفيف", "خفيفة", "بسيط"]):
            severity = "mild"
        elif any(t in text for t in ["متوسط", "متوسطة"]):
            severity = "moderate"
        elif any(t in text for t in ["شديد", "شديدة", "حاد", "قوي", "لا يحتمل"]):
            severity = "severe"

        pain_score = None
        for i in range(11):
            if str(i) in text:
                pain_score = i
                break

        return {
            "yes": any(t in text for t in ["نعم", "ايوه", "يوجد", "موجود"]),
            "severe": severity == "severe",
            "moderate": severity == "moderate",
            "persistent": any(t in text for t in ["مستمر", "دائم", "طول الوقت", "ثابت"]),
            "urgent": any(t in text for t in ["اغماء", "دوخه", "ضيق تنفس", "نزيف"]),
            "radiating": any(t in text for t in ["يمتد", "ينتشر", "الى الذراع", "إلى الذراع", "للفك", "للظهر"]),
            "numbness": any(t in text for t in ["تنميل", "خدر", "وخز"]),
            "movement_worse": any(t in text for t in ["يزداد", "تزداد", "مع الحركة", "عند الحركة"]),
            "weakness": any(t in text for t in ["ضعف", "لا استطيع", "لا أستطيع", "عدم القدرة"]),
            "worsening": any(t in text for t in ["تزداد", "يسوء", "تسوء"]),
            "duration": duration,
            "pain_score": pain_score,
            "diabetes": "سكري" in text or "السكري" in text,
            "pressure": "ضغط" in text,
            "heart_disease": "قلب" in text or "قلبي" in text,
        }

    @staticmethod
    def confidence_band(conf: float) -> str:
        if conf >= 0.80:
            return "high confidence"
        elif conf >= 0.60:
            return "moderate confidence"
        return "low confidence"

    @staticmethod
    def timing_from_risk(risk_level: str) -> str:
        if risk_level == "high":
            return "مراجعة اليوم"
        elif risk_level == "medium":
            return "حجز موعد قريب"
        return "مراقبة قصيرة"

    @staticmethod
    def self_care_advice(specialty: str) -> List[str]:
        advice = [
            "الراحة وتجنب الإجهاد الشديد.",
            "شرب كمية كافية من السوائل إذا لم يوجد مانع طبي.",
            "تجنب حمل الأشياء الثقيلة حتى التقييم الطبي.",
        ]
        if specialty == "جراحة العظام والمفاصل":
            advice.append("تقليل الحركات المجهدة للمنطقة المصابة.")
        return advice

    def generate(
        self,
        specialty: str,
        qa_pairs: List[Tuple[QuestionTemplate, str]],
        safety_result: Optional[Dict[str, str]] = None,
    ) -> Dict[str, object]:
        if safety_result is not None:
            timing = "طوارئ الآن" if safety_result["level"] == "emergency" else "مراجعة اليوم"
            return {
                "risk_level": safety_result["level"],
                "score": 999,
                "timing": timing,
                "advice": [
                    safety_result["reason"],
                    safety_result["advice"],
                    "هذا النظام لا يقدم تشخيصاً نهائياً."
                ],
                "rationale": [safety_result["reason"]],
                "self_care": [],
            }

        score = 0
        rationale = []

        for question, answer in qa_pairs:
            features = self._answer_features(answer)

            if features["yes"]:
                score += question.risk_weight_yes
                rationale.append(f"إجابة إيجابية على {question.category}")

            if features["severe"]:
                score += question.risk_weight_severe + 1
                rationale.append(f"شدة مرتفعة في {question.category}")

            if features["moderate"]:
                score += 1

            if features["persistent"]:
                score += 1

            if features["urgent"]:
                score += 4
                rationale.append("تم ذكر علامة إنذارية في الإجابات")

            if features["radiating"]:
                score += 2
                rationale.append("تم ذكر امتداد الألم")

            if features["numbness"]:
                score += 3
                rationale.append("تم ذكر تنميل أو خدر")

            if features["movement_worse"]:
                score += 2
                rationale.append("الأعراض تزداد مع الحركة")

            if features["weakness"]:
                score += 4
                rationale.append("تم ذكر ضعف أو عدم قدرة وظيفية")

            if features["worsening"]:
                score += 2
                rationale.append("الأعراض تتفاقم")

            if features["pain_score"] is not None and features["pain_score"] >= 7:
                score += 2
                rationale.append("درجة الألم مرتفعة")
            elif features["pain_score"] is not None and features["pain_score"] >= 4:
                score += 1

            if features["duration"] in ["acute", "subacute"]:
                score += 1

            if features["diabetes"] or features["pressure"] or features["heart_disease"]:
                score += 2
                rationale.append("يوجد عامل مرض مزمن يحتاج حذراً أكبر")

        if score >= 9:
            risk_level = "high"
        elif score >= 4:
            risk_level = "medium"
        else:
            risk_level = "low"

        timing = self.timing_from_risk(risk_level)

        advice = [f"التخصص الأكثر ملاءمة حالياً: {specialty}"]

        if specialty == "جراحة العظام والمفاصل":
            advice.append("الترجيح يميل لمشكلة مفصلية أو عضلية أو متعلقة بالأوتار.")
        elif specialty == "طب الاعصاب":
            advice.append("الترجيح يميل لمسبب عصبي أو ضغط على الأعصاب.")
        elif specialty == "أمراض القلب":
            advice.append("الأعراض الصدرية والتنفسية تستدعي اهتماماً خاصاً وسريعاً.")
        else:
            advice.append("الحالة تحتاج تقييماً سريرياً للوصول إلى التشخيص النهائي.")

        advice.append(f"التوصية الزمنية: {timing}")
        advice.append("هذا النظام أداة دعم قرار أولية وليس بديلاً عن الطبيب.")

        return {
            "risk_level": risk_level,
            "score": score,
            "timing": timing,
            "rationale": rationale,
            "advice": advice,
            "self_care": self.self_care_advice(specialty),
        }


recommendation_engine = RecommendationEngine()


Explanation

This layer explains not only which words influenced the prediction, but also how the two branches of the hybrid model behaved and which similar historical cases supported the final decision. This is much more presentation-ready than standard LIME alone.

In [ ]:
class ExplainabilityLayer:
    def __init__(self, preprocessor: ArabicPreprocessor, hybrid_engine: HybridDecisionEngine, config: SystemConfig, symptom_extractor: SymptomExtractor):
        self.preprocessor = preprocessor
        self.hybrid_engine = hybrid_engine
        self.config = config
        self.symptom_extractor = symptom_extractor
        self.explainer = LimeTextExplainer(class_names=hybrid_engine.all_labels)

    def hybrid_predict_proba(self, texts: List[str]) -> np.ndarray:
        rows = []
        for text in texts:
            extracted = self.symptom_extractor.extract(text)
            result = self.hybrid_engine.predict(text, extracted=extracted)
            rows.append(result["final_probs"])
        return np.vstack(rows)

    def explain(self, text: str) -> Dict[str, object]:
        clean_text = self.preprocessor.preprocess(text)
        extracted = self.symptom_extractor.extract(text)
        result = self.hybrid_engine.predict(text, extracted=extracted)

        exp = self.explainer.explain_instance(
            text_instance=clean_text,
            classifier_fn=self.hybrid_predict_proba,
            num_features=self.config.lime_num_features,
        )

        why_won = (
            f"تم ترجيح {result['final_label']} على {result['second_label']} "
            f"لأن احتماله النهائي أعلى بفارق {result['confidence_gap']:.2f}"
        )

        retrieved_summary = [
            {
                "label": item["label"],
                "score": round(item["final_score"], 4),
                "text_preview": item["text"][:150]
            }
            for item in result["retrieved_cases"][:3]
        ]

        return {
            "predicted_specialty": result["final_label"],
            "raw_confidence": result["final_confidence_raw"],
            "word_importance": exp.as_list(),
            "retrieved_support": retrieved_summary
        }


Explanation

This class integrates all system components into one professional pipeline. It controls the flow of preprocessing, extraction, safety screening, classification, retrieval, question generation, recommendation, and explainability.

In [ ]:
class ArabicMedicalTriageSystem:
    def __init__(
        self,
        config: SystemConfig,
        preprocessor: ArabicPreprocessor,
        symptom_extractor: SymptomExtractor,
        safety_layer: SafetyLayer,
        classifier,
        retriever: DenseArabicRetriever,
        hybrid_engine: HybridDecisionEngine,
        calibrator: ConfidenceCalibrator,
        question_engine: AdaptiveQuestionEngine,
        recommendation_engine: RecommendationEngine,
        explainability_layer: ExplainabilityLayer,
    ):
        self.config = config
        self.preprocessor = preprocessor
        self.symptom_extractor = symptom_extractor
        self.safety_layer = safety_layer
        self.classifier = classifier
        self.retriever = retriever
        self.hybrid_engine = hybrid_engine
        self.calibrator = calibrator
        self.question_engine = question_engine
        self.recommendation_engine = recommendation_engine
        self.explainability_layer = explainability_layer
        self.case_logs = []

    def predict(self, user_text: str) -> Dict[str, object]:
        extracted = self.symptom_extractor.extract(user_text)
        safety = self.safety_layer.screen(user_text, extracted)

        if safety and safety["level"] == "emergency":
            return {
                "mode": "emergency",
                "triage_category": "emergency",
                "safety": safety,
                "extracted": extracted
            }

        result = self.hybrid_engine.predict(user_text, extracted=extracted)
        calibrated_conf = self.calibrator.calibrate(result["final_confidence_raw"])

        result["final_confidence"] = calibrated_conf
        result["confidence_band"] = self.recommendation_engine.confidence_band(calibrated_conf)
        result["extracted"] = extracted
        result["safety"] = safety
        result["mode"] = "normal"

        if result["confidence_gap"] < 0.10 or result["final_confidence"] < 0.55:
            result["is_uncertain"] = True
            result["uncertainty_message"] = "الحالة غير محسومة بالكامل، ويفضل دعم القرار بالفحص السريري ومزيد من الأسئلة."
        else:
            result["is_uncertain"] = False
            result["uncertainty_message"] = None

        result["triage_category"] = safety["level"] if safety else "non-urgent"
        return result

    def generate_questions(self, user_text: str) -> List[QuestionTemplate]:
        prediction = self.predict(user_text)
        if prediction["mode"] == "emergency":
            return []

        return self.question_engine.select_questions(
            user_text=user_text,
            predicted_specialty=prediction["final_label"],
            final_confidence=prediction["final_confidence"],
            extracted=prediction["extracted"],
            max_questions=self.config.max_follow_up_questions,
        )

    def recommend(self, user_text: str, answers: List[str]) -> Dict[str, object]:
        prediction = self.predict(user_text)

        if prediction["mode"] == "emergency":
            return self.recommendation_engine.generate(
                specialty="Emergency",
                qa_pairs=[],
                safety_result=prediction["safety"],
            )

        questions = self.generate_questions(user_text)
        qa_pairs = list(zip(questions, answers))
        recommendation = self.recommendation_engine.generate(
            specialty=prediction["final_label"],
            qa_pairs=qa_pairs,
            safety_result=prediction["safety"],
        )

        return {
            "prediction": prediction,
            "recommendation": recommendation,
            "qa_pairs": qa_pairs,
        }

    def build_case_summary(self, user_text: str, answers: List[str]) -> Dict[str, object]:
        payload = self.recommend(user_text, answers)

        if "prediction" not in payload:
            return {"mode": "emergency", "summary_lines": payload["advice"]}

        prediction = payload["prediction"]
        recommendation = payload["recommendation"]
        extracted = prediction["extracted"]

        summary_lines = [
            f"الشكوى: {user_text}",
            f"فئة الفرز: {prediction['triage_category']}",
            f"التخصص المتوقع: {prediction['final_label']}",
            f"الثقة: {prediction['confidence_band']} ({prediction['final_confidence']:.2f})",
            f"الموجودات المستخرجة: أجزاء الجسم={extracted['body_parts']}, الشدة={extracted['severity']}, المدة={extracted['duration']}, الانتشار={extracted['spread']}",
            f"العلامات الإنذارية: {', '.join(extracted['red_flags']) if extracted['red_flags'] else 'لا يوجد واضح'}",
            f"الخطوة التالية: {recommendation['timing']}",
        ]
        return {"mode": "normal", "summary_lines": summary_lines}

    def log_case(self, user_text: str, result: Dict[str, object]):
        if "prediction" not in result:
            return
        prediction = result["prediction"]
        recommendation = result["recommendation"]
        self.case_logs.append({
            "complaint": user_text,
            "predicted_specialty": prediction["final_label"],
            "triage_category": prediction["triage_category"],
            "confidence": prediction["final_confidence"],
            "confidence_band": prediction["confidence_band"],
            "risk_level": recommendation["risk_level"],
            "timing": recommendation["timing"],
        })

    def export_case_logs(self) -> pd.DataFrame:
        return pd.DataFrame(self.case_logs)

    def explain(self, user_text: str) -> Dict[str, object]:
        return self.explainability_layer.explain(user_text)


In [ ]:
def evaluation_dashboard(system, test_df):
    true_labels = test_df["label"].tolist()
    preds = []
    retrieval_hits = []

    for text, true_label in zip(test_df["clean"].tolist(), true_labels):
        out = system.predict(text)
        if out["mode"] == "emergency":
            continue
        preds.append(out["final_label"])
        retrieved_labels = [x["label"] for x in out["retrieved_cases"]]
        retrieval_hits.append(int(true_label in retrieved_labels))

    used_true = true_labels[:len(preds)]

    print("Overall Accuracy:", accuracy_score(used_true, preds))
    print("Macro F1:", f1_score(used_true, preds, average="macro"))
    print("Retrieval Hit Rate:", np.mean(retrieval_hits))

    print("\nPer-class F1:")
    p, r, f, s = precision_recall_fscore_support(
        used_true,
        preds,
        average=None,
        labels=sorted(set(used_true))
    )
    for label, score in zip(sorted(set(used_true)), f):
        print(f"- {label}: {score:.3f}")

    cm = pd.DataFrame(
        confusion_matrix(used_true, preds),
        index=sorted(set(used_true)),
        columns=sorted(set(used_true))
    )

    print("\nMost confused specialties:")
    confusion_pairs = []
    for i, row_label in enumerate(cm.index):
        for j, col_label in enumerate(cm.columns):
            if i != j and cm.iloc[i, j] > 0:
                confusion_pairs.append((row_label, col_label, cm.iloc[i, j]))

    confusion_pairs = sorted(confusion_pairs, key=lambda x: x[2], reverse=True)[:10]
    for a, b, v in confusion_pairs:
        print(f"- {a} -> {b}: {v}")


Explanation

This cell instantiates the final end-to-end system. From this point onward, all components are connected and ready for evaluation and interaction.

In [ ]:
hybrid_engine = HybridDecisionEngine(arabert_classifier, retriever)

explainability_layer = ExplainabilityLayer(
    preprocessor=preprocessor,
    hybrid_engine=hybrid_engine,
    config=config,
    symptom_extractor=symptom_extractor
)

medical_system = ArabicMedicalTriageSystem(
    config=config,
    preprocessor=preprocessor,
    symptom_extractor=symptom_extractor,
    safety_layer=safety_layer,
    classifier=arabert_classifier,
    retriever=retriever,
    hybrid_engine=hybrid_engine,
    calibrator=confidence_calibrator,
    question_engine=question_engine,
    recommendation_engine=recommendation_engine,
    explainability_layer=explainability_layer,
)


Explanation

This is a stronger evaluation protocol. It measures not just classification accuracy, but also top-3 accuracy, retrieval hit rate, and MRR. This makes the project much more research-oriented and much more defendable in front of a supervisor.

In [ ]:
def evaluate_hybrid_system(system: ArabicMedicalTriageSystem, test_df: pd.DataFrame) -> Dict[str, object]:
    true_labels = test_df["label"].tolist()
    texts = test_df["clean"].tolist()

    final_probs = []
    final_preds = []

    retrieval_hits = []
    reciprocal_ranks = []

    for text, true_label in zip(texts, true_labels):
        pred = system.predict(text)

        if pred["mode"] == "emergency":
            continue

        probs = pred["final_probs"]
        final_probs.append(probs)
        final_preds.append(pred["final_label"])

        retrieved_labels = [x["label"] for x in pred["retrieved_cases"]]
        retrieval_hits.append(int(true_label in retrieved_labels))

        if true_label in retrieved_labels:
            rank = retrieved_labels.index(true_label) + 1
            reciprocal_ranks.append(1.0 / rank)
        else:
            reciprocal_ranks.append(0.0)

    final_probs = np.vstack(final_probs)
    label_space = hybrid_engine.all_labels
    label_to_idx = {label: i for i, label in enumerate(label_space)}
    y_true_idx = np.array([label_to_idx[y] for y in true_labels[:len(final_preds)]])

    results = {
        "accuracy": accuracy_score(true_labels[:len(final_preds)], final_preds),
        "macro_f1": f1_score(true_labels[:len(final_preds)], final_preds, average="macro"),
        "top_3_accuracy": top_k_accuracy_score(
            y_true_idx,
            final_probs,
            k=min(3, final_probs.shape[1]),
            labels=np.arange(final_probs.shape[1]),
        ),
        "retrieval_hit_at_k": float(np.mean(retrieval_hits)),
        "retrieval_mrr": float(np.mean(reciprocal_ranks)),
        "confusion_matrix": confusion_matrix(true_labels[:len(final_preds)], final_preds),
        "classification_report": classification_report(true_labels[:len(final_preds)], final_preds),
    }
    return results


final_results = evaluate_hybrid_system(medical_system, test_df)
print("Hybrid Accuracy:", final_results["accuracy"])
print("Hybrid Macro F1:", final_results["macro_f1"])
print("Hybrid Top-3 Accuracy:", final_results["top_3_accuracy"])
print("Retriever Hit@K:", final_results["retrieval_hit_at_k"])
print("Retriever MRR:", final_results["retrieval_mrr"])
print(final_results["classification_report"])


Hybrid Accuracy: 0.2165097755249819
Hybrid Macro F1: 0.20097378899661272
Hybrid Top-3 Accuracy: 0.6188993482983346
Retriever Hit@K: 0.8741491672700942
Retriever MRR: 0.7243905382573015
                       precision    recall  f1-score   support

          الطب النفسي       0.20      0.18      0.19      2781
                تغذية       0.16      0.15      0.16      2059
جراحة العظام والمفاصل       0.28      0.35      0.31      3803
             طب اسنان       0.20      0.16      0.18      2783
              طب عيون       0.18      0.17      0.17      2384

             accuracy                           0.22     13810
            macro avg       0.20      0.20      0.20     13810
         weighted avg       0.21      0.22      0.21     13810



Explanation

Error analysis is a professional touch. It shows where the model fails, which classes are confused, and how uncertain the system was. This is often one of the most valuable parts in a thesis discussion chapter.

In [ ]:
def error_analysis(system: ArabicMedicalTriageSystem, test_df: pd.DataFrame, max_examples: int = 10):
    errors = []

    for _, row in test_df.iterrows():
        text = row["clean"]
        true_label = row["label"]
        pred = system.predict(text)

        if pred["mode"] == "normal" and pred["final_label"] != true_label:
            errors.append({
                "text": row["text"][:200],
                "true_label": true_label,
                "pred_label": pred["final_label"],
                "confidence": pred["final_confidence"],
            })

    error_df = pd.DataFrame(errors)
    return error_df.head(max_examples)


error_examples = error_analysis(medical_system, test_df, max_examples=10)
error_examples


,text,true_label,pred_label,confidence
0,الم في الصدر و تعب من المجهود البسيط,جراحة العظام والمفاصل,طب عيون,0.490566
1,………………..٨٨٨٨,الطب النفسي,طب عيون,0.203390
2,مساء الخير انا حشيت ضرسي من فترة وفيه تورم بسي...,طب اسنان,جراحة العظام والمفاصل,0.801932
3,أريد دكتورة تغذية,تغذية,جراحة العظام والمفاصل,0.624088
4,السلام عليكم انا طولي 189 ووزني 82 كنت محتاج ا...,تغذية,جراحة العظام والمفاصل,0.940403
5,السلام عليكم انا اعاني من وزن زائد طولي 143ووز...,تغذية,جراحة العظام والمفاصل,0.624088
6,اريد استخدام حبوب فاتلوس لخسارة الوزن ولدى كسل...,تغذية,جراحة العظام والمفاصل,0.801932
7,طبيب المسالك البوليه,تغذية,الطب النفسي,0.076923
8,امزاض القلب,طب عيون,الطب النفسي,0.372000
9,اعاني من شد في منطقه اعلى الصدر من جهه اليمين ...,الطب النفسي,جراحة العظام والمفاصل,0.993080


Explanation

This final chatbot makes the system feel much more mature. It shows the primary and secondary specialties, calibrated confidence, extracted symptom structure, adaptive follow-up, final recommendation, and optional explainability.

In [ ]:
class ArabicUX:
    @staticmethod
    def intro():
        print("\nمرحباً بك.")
        print("هذا النظام يساعد في التوجيه الطبي الأولي بناءً على الأعراض، لكنه لا يضع تشخيصاً نهائياً.")
        print("يمكنك كتابة exit في أي وقت للخروج.\n")

    @staticmethod
    def ask_main_complaint():
        return input("من فضلك اكتب الشكوى أو الأعراض الرئيسية:\n> ").strip()

    @staticmethod
    def section(title: str):
        print(f"\n{title}")

    @staticmethod
    def gentle_emergency_message(safety: Dict[str, str]):
        print("\nتنبيه مهم يتعلق بالسلامة:")
        print(f"- السبب: {safety['reason']}")
        print(f"- التوصية: {safety['advice']}")
        print("- من الأفضل عدم الاعتماد على النظام وحده في هذه الحالة.")

    @staticmethod
    def show_triage(pred: Dict[str, object]):
        print("\nنتيجة الفرز الأولي:")
        print(f"- فئة الفرز: {pred['triage_category']}")
        print(f"- التخصص المتوقع: {pred['final_label']}")
        print(f"- التخصص البديل: {pred['second_label']}")
        print(f"- مستوى الثقة: {pred['confidence_band']} ({pred['final_confidence']:.2f})")
        print(f"- الفارق بين الأول والثاني: {pred['confidence_gap']:.2f}")

        if pred.get("is_uncertain", False):
            print(f"- ملاحظة: {pred['uncertainty_message']}")

    @staticmethod
    def show_extracted_features(pred: Dict[str, object]):
        extracted = pred["extracted"]
        print("\nالملامح المستخرجة من الشكوى:")
        print(f"- أجزاء الجسم: {extracted['body_parts'] if extracted['body_parts'] else ['غير محدد']}")
        print(f"- الشدة: {extracted['severity']}")
        print(f"- المدة: {extracted['duration']}")
        print(f"- الانتشار: {extracted['spread']}")
        print(f"- العلامات الإنذارية: {extracted['red_flags'] if extracted['red_flags'] else ['لا يوجد واضح']}")

    @staticmethod
    def show_urgent_reason(pred: Dict[str, object]):
        if pred.get("safety") is not None and pred["safety"]["level"] == "urgent":
            print("\nسبب التنبيه:")
            print(f"- {pred['safety']['reason']}")
            print(f"- {pred['safety']['advice']}")

    @staticmethod
    def explain_why(question_obj):
        if getattr(question_obj, "why", ""):
            print(f"سبب السؤال: {question_obj.why}")

    @staticmethod
    def before_questions():
        print("\nسأطرح عليك بعض الأسئلة القصيرة لتحسين التوجيه الطبي.")

    @staticmethod
    def escalation_message():
        print("\nيوجد ارتفاع في مستوى الخطورة بناءً على الإجابات الحالية.")
        print("- يُفضّل التقييم الطبي بشكل عاجل وعدم تأجيل المتابعة.")

    @staticmethod
    def show_recommendation(recommendation: Dict[str, object]):
        print("\nالتوصية النهائية:")
        print(f"- مستوى الخطورة: {recommendation['risk_level']}")
        print(f"- الدرجة: {recommendation['score']}")
        print(f"- التوقيت المقترح: {recommendation['timing']}")

        if recommendation.get("rationale"):
            print("- أسباب دعمت القرار:")
            for reason in recommendation["rationale"][:5]:
                print(f"  {reason}")

        if recommendation.get("self_care"):
            print("- نصائح عامة آمنة:")
            for item in recommendation["self_care"]:
                print(f"  {item}")

        print("- الخلاصة:")
        for line in recommendation["advice"]:
            print(f"  {line}")

    @staticmethod
    def show_summary(summary: Dict[str, object]):
        if summary["mode"] == "normal":
            print("\nملخص الحالة:")
            for line in summary["summary_lines"]:
                print(f"- {line}")

    @staticmethod
    def show_explanation(exp: Dict[str, object]):
        print("\nتفسير القرار:")
        print(f"- التخصص المتوقع: {exp['predicted_specialty']}")
        print(f"- التخصص البديل: {exp['second_specialty']}")
        print(f"- لماذا تم ترجيحه؟ {exp['why_top1_beat_top2']}")
        print("- أهم الكلمات:")
        for word, weight in exp["word_importance"]:
            print(f"  {word}: {weight:.4f}")
        print("- أقرب الحالات المسترجعة:")
        for item in exp["retrieved_support"]:
            print(f"  [{item['label']}] score={item['score']} | {item['text_preview']}")

    @staticmethod
    def closing():
        print("\nانتهت الجلسة. نتمنى لك السلامة.\n")


In [ ]:
def get_supportive_optional_tips(specialty: str, risk_level: str) -> List[str]:
    if risk_level in ["high", "emergency", "urgent"]:
        return []

    tips = [
        "هذه إرشادات داعمة اختيارية فقط ولا تغني عن مراجعة الطبيب.",
        "إذا ساءت الأعراض أو ظهرت علامات جديدة، يجب طلب تقييم طبي مباشرة."
    ]

    if specialty == "جراحة العظام والمفاصل":
        tips.extend([
            "يمكن الاستفادة من الراحة المؤقتة وتقليل الحركات المجهدة.",
            "قد تساعد الكمادات الباردة أو الدافئة بحسب ما يريحك.",
            "يمكن ممارسة تمارين تمدد خفيفة فقط إذا لم تسبب زيادة في الألم.",
        ])
    elif specialty == "طب الاعصاب":
        tips.extend([
            "قد يفيد تقليل الإجهاد الجسدي والنفسي والنوم الجيد.",
            "يفضل تجنب الجلسات أو الوضعيات التي تزيد التنميل أو الألم.",
            "يمكن استخدام وسائل استرخاء خفيفة مثل التنفس العميق أو الاسترخاء التدريجي.",
        ])
    elif specialty == "الباطنية":
        tips.extend([
            "قد يساعد شرب السوائل وتنظيم الوجبات الخفيفة إذا كانت الحالة تسمح بذلك.",
            "يفضل تجنب الأطعمة التي تلاحظ أنها تزيد الأعراض.",
            "الراحة ومراقبة تطور الأعراض قد يكونان مفيدين إلى حين المراجعة الطبية.",
        ])
    elif specialty == "أمراض القلب":
        tips.extend([
            "في الأعراض الخفيفة فقط: يفضل تقليل الجهد البدني والراحة إلى حين التقييم الطبي.",
            "إذا ظهر ألم صدر شديد أو ضيق تنفس أو تعرق شديد، فلا تعتمد على هذه النصائح واطلب المساعدة الطبية فوراً.",
        ])
    else:
        tips.extend([
            "يمكن الاكتفاء بالراحة، شرب السوائل، ومراقبة الأعراض إلى حين مراجعة الطبيب.",
        ])

    return tips


In [ ]:
def run_interactive_chat(system: ArabicMedicalTriageSystem, mode="doctor"):
    ArabicUX.intro()

    while True:
        user_text = input("من فضلك اكتب الشكوى أو الأعراض الرئيسية:\n> ").strip()

        if user_text.lower() == "exit":
            print("تم إنهاء الجلسة.")
            break

        pred = system.predict(user_text)

        if pred["mode"] == "emergency":
            print("\nتنبيه سلامة:")
            print(f"- {pred['safety']['reason']}")
            print(f"- {pred['safety']['advice']}")
            print("- هذا النظام لا يغني عن التقييم الطبي المباشر.")
            print("\n" + "=" * 70 + "\n")
            continue

        print("\nنتيجة الفرز الأولي:")
        print(f"- فئة الفرز: {pred['triage_category']}")
        print(f"- التخصص المتوقع: {pred['final_label']}")
        print(f"- مستوى الثقة: {pred['confidence_band']} ({pred['final_confidence']:.2f})")


        if pred.get("is_uncertain", False):
            print(f"- ملاحظة: {pred['uncertainty_message']}")
            print("- سيتم طرح أسئلة إضافية قبل تثبيت التوصية.")

        print("\nالملامح المستخرجة:")
        print(f"- أجزاء الجسم: {pred['extracted']['body_parts'] if pred['extracted']['body_parts'] else ['غير محدد']}")
        print(f"- الشدة: {pred['extracted']['severity']}")
        print(f"- المدة: {pred['extracted']['duration']}")
        print(f"- الانتشار: {pred['extracted']['spread']}")

        if pred["safety"] is not None and pred["safety"]["level"] == "urgent":
            print("\nسبب التنبيه:")
            print(f"- {pred['safety']['reason']}")
            print(f"- {pred['safety']['advice']}")

        questions = system.generate_questions(user_text)
        answers = []

        print("\nأسئلة المتابعة:")
        for q in questions:
            if mode == "patient" and getattr(q, "why", ""):
                print(f"سبب السؤال: {q.why}")

            ans = input(f"{q.question}\n> ").strip()
            answers.append(ans)

            interim = system.recommendation_engine.generate(
                specialty=pred["final_label"],
                qa_pairs=list(zip(questions[:len(answers)], answers)),
                safety_result=pred["safety"],
            )

            if interim["risk_level"] == "high":
                print("- تم رفع الحالة إلى خطورة عالية بناءً على الإجابات الحالية.")
                print("- يفضل التقييم الطبي العاجل اليوم.")
                break

        result = system.recommend(user_text, answers)
        recommendation = result["recommendation"] if "recommendation" in result else result

        if isinstance(result, dict) and "prediction" in result:
            system.log_case(user_text, result)

        print("\nالتوصية النهائية:")
        print(f"- مستوى الخطورة: {recommendation['risk_level']}")
        print(f"- الدرجة: {recommendation['score']}")
        print(f"- التوقيت المقترح: {recommendation['timing']}")

        print("- نصائح عامة آمنة:")
        for item in recommendation.get("self_care", []):
            print(f"  {item}")

        if recommendation.get("rationale"):
            print("- أسباب دعمت القرار:")
            for reason in recommendation["rationale"][:5]:
                print(f"  {reason}")

        print("- الخلاصة:")
        for line in recommendation["advice"]:
            print(f"  {line}")

        summary = system.build_case_summary(user_text, answers)
        if summary["mode"] == "normal":
            print("\nملخص الطبيب/المشرف:")
            for line in summary["summary_lines"]:
                print(f"- {line}")

        show_exp = input("\nهل تريد تفسير القرار؟ (y/n)\n> ").strip().lower()
        if show_exp == "y":
            exp = system.explain(user_text)
            print("\nالتفسير:")
            print(f"- التخصص المتوقع: {exp['predicted_specialty']}")
            print("- أهم الكلمات:")
            for word, weight in exp["word_importance"]:
                print(f"  {word}: {weight:.4f}")
            print("- أقرب الحالات المسترجعة:")
            for item in exp["retrieved_support"]:
                print(f"  [{item['label']}] score={item['score']} | {item['text_preview']}")

        show_supportive = input("\nهل تريد عرض إرشادات داعمة اختيارية إلى حين مراجعة الطبيب؟ (y/n)\n> ").strip().lower()
        if show_supportive == "y":
            supportive_tips = get_supportive_optional_tips(
                specialty=pred["final_label"],
                risk_level=recommendation["risk_level"]
            )

            if supportive_tips:
                print("\nإرشادات داعمة اختيارية:")
                for tip in supportive_tips:
                    print(f"- {tip}")
            else:
                print("\nلا يُنصح بعرض إرشادات داعمة في هذه الحالة لأن مستوى الخطورة أعلى ويجب التركيز على التقييم الطبي.")

        print("\n" + "=" * 70 + "\n")


Explanation

This cell launches the interactive triage assistant for live use in Colab.

 ألم بالكتف الايسر من فترة

In [ ]:
run_interactive_chat(medical_system, mode="doctor")


مرحباً بك.
هذا النظام يساعد في التوجيه الطبي الأولي بناءً على الأعراض، لكنه لا يضع تشخيصاً نهائياً.
يمكنك كتابة exit في أي وقت للخروج.

من فضلك اكتب الشكوى أو الأعراض الرئيسية:
>  ألم بالكتف الايسر من فترة

نتيجة الفرز الأولي:
- فئة الفرز: non-urgent
- التخصص المتوقع: جراحة العظام والمفاصل
- مستوى الثقة: high confidence (0.99)

الملامح المستخرجة:
- أجزاء الجسم: ['كتف']
- الشدة: unknown
- المدة: unknown
- الانتشار: no

أسئلة المتابعة:
كم عمرك تقريباً؟ (طفل / بالغ / كبير سن)
> بالغ 
ما الجنس؟ (ذكر / أنثى)
> ذكر 
هل لديك أمراض مزمنة مثل السكري أو الضغط أو أمراض القلب؟
> الضغط 
أين مكان الألم أو العرض بشكل أدق؟
> كتف
منذ متى بدأت الأعراض؟
> > منذ أسبوع
من 0 إلى 10 كم شدة الألم؟
> 5
هل الأعراض تتحسن، ثابتة، أم تزداد؟
> نعم
هل يزيد الألم مع الحركة؟
> نعم
هل يمتد الألم إلى الذراع؟
> نعم
هل يوجد تنميل أو خدر في الذراع أو اليد؟
> نعم

التوصية النهائية:
- مستوى الخطورة: medium
- الدرجة: 8
- التوقيت المقترح: حجز موعد قريب
- نصائح عامة آمنة:
  الراحة وتجنب الإجهاد الشديد.
  شرب كمية كافية من السو

In [ ]:
logs_df = medical_system.export_case_logs()
logs_df.head()


,complaint,predicted_specialty,triage_category,confidence,confidence_band,risk_level,timing
0,ألم بالكتف الايسر من فترة,جراحة العظام والمفاصل,non-urgent,0.988364,high confidence,medium,حجز موعد قريب


In [ ]:
evaluation_dashboard(medical_system, test_df)


Overall Accuracy: 0.2165097755249819
Macro F1: 0.20097378899661272
Retrieval Hit Rate: 0.8741491672700942

Per-class F1:
- الطب النفسي: 0.192
- تغذية: 0.155
- جراحة العظام والمفاصل: 0.307
- طب اسنان: 0.179
- طب عيون: 0.171

Most confused specialties:
- الطب النفسي -> جراحة العظام والمفاصل: 1003
- طب اسنان -> جراحة العظام والمفاصل: 960
- طب عيون -> جراحة العظام والمفاصل: 801
- جراحة العظام والمفاصل -> الطب النفسي: 708
- تغذية -> جراحة العظام والمفاصل: 675
- جراحة العظام والمفاصل -> طب اسنان: 675
- جراحة العظام والمفاصل -> طب عيون: 584
- جراحة العظام والمفاصل -> تغذية: 521
- طب اسنان -> الطب النفسي: 521
- طب اسنان -> طب عيون: 451


Thesis-Style Summary of the Full Project
This project proposes a professional Arabic medical triage and specialty recommendation system designed as a decision-support tool rather than a diagnosis system. The architecture combines Arabic text preprocessing, classical baselines, AraBERT-based semantic classification, dense retrieval over historical cases, a confidence-aware hybrid fusion mechanism, structured symptom extraction, adaptive follow-up questioning, risk-based recommendation generation, emergency safety screening, and explainability using LIME.

The preprocessing module normalizes Arabic orthography and removes linguistic noise to improve both classification and retrieval. The classifier module uses AraBERT as a frozen encoder with a lightweight logistic regression head, which offers a practical balance between semantic performance and Colab efficiency. The retrieval module builds a balanced FAISS index over training data only, then combines dense semantic similarity with lexical reranking to produce more grounded and clinically interpretable retrieval results.

The hybrid decision layer integrates classifier probabilities with retrieval evidence using dynamic weighting based on model confidence and branch agreement. To improve reliability, a confidence calibration stage is learned on the validation set using isotonic regression. This ensures that the final confidence score is more meaningful and suitable for decision-support scenarios.

A structured symptom extraction layer identifies body parts, severity, duration, and red-flag symptoms directly from the patient narrative. These extracted features guide both the safety layer and the adaptive question engine. The safety layer detects emergency or urgent patterns before normal triage proceeds, ensuring clinically safer behavior. The chatbot then asks follow-up questions dynamically rather than using a fixed list, making the system more intelligent and context-aware.

The recommendation engine analyzes the actual question-answer pairs and converts them into a risk level and triage recommendation. Finally, the explainability layer provides a presentation-ready explanation by combining LIME feature attribution with retrieved supporting examples and branch-level hybrid reasoning.

From a research perspective, the system is evaluated not only with classification accuracy and macro-F1, but also with top-k accuracy, retrieval hit rate, mean reciprocal rank, and error analysis. This makes the project stronger academically and more convincing as a final-year AI system.